<a href="https://colab.research.google.com/github/SREEJANIIII/react-agent-demo/blob/main/ReAct_Agent_Demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI Agent Loop Demo


This notebook demonstrates the core mechanism behind an AI agent: a model that can decide to call a tool, receive the result, and continue reasoning toward a goal — all in a loop.

---

This notebook contains starter code. Please fill in details as needed.
---

## Setup - Gemini API key

How to get the key (takes about two minutes):

* Go to [aistudio.google.com](https://aistudio.google.com/prompts/new_chat)
* Sign in with your Google account
* Click "Get API key" in the left sidebar
* Click "Create API key" — choose "Create API key in new project" if prompted
Copy the key that appears (it starts with AIza...)

> The free tier allows up to 10 requests per minute and 250 requests per day, which is enough to work through this notebook.

If you see occasional slow-downs when you run several questions back to back — that is normal, and the code handles it automatically.

## References
* [Google agent loop](https://github.com/xiaosuhu/midas-ai-in-research/blob/v1.0-dev/docs/notebooks/agent_loop_demo.ipynb$0)
* https://ai.google.dev/gemini-api/docs/thinking#signatures
* https://ai.google.dev/gemini-api/docs/text-generation$0
* [Developer guide](https://github.com/xiaosuhu/midas-ai-in-research/blob/v1.0-dev/docs/notebooks/agent_loop_demo.ipynb$0)



In [2]:
import os
from google.colab import userdata

# Load Gemini API key securely from Colab Secrets
try:
    api_key = userdata.get("GOOGLE_API_KEY")

    if not api_key:
        raise ValueError("API key is empty or unavailable.")

    os.environ["GOOGLE_API_KEY"] = api_key
    print("API key loaded successfully from Colab Secrets.")

except Exception as e:
    import getpass
    api_key = getpass.getpass("Enter your Gemini API key: ")

    if not api_key.strip():
        raise ValueError("API key cannot be empty.")

    os.environ["GOOGLE_API_KEY"] = api_key
    print("API key configured successfully.")

API key loaded successfully from Colab Secrets.


In [3]:
from google import genai
import os

client = genai.Client(api_key=os.environ["GOOGLE_API_KEY"])

print("Gemini client initialized successfully!")

Gemini client initialized successfully!


In [2]:
# Remove the outdated Gemini SDK
!pip uninstall -y -q google-generativeai

# Install compatible Gemini SDK and dependencies
!pip install -q "google-genai>=2.12.1,<3" \
    "protobuf>=6.33.5,<8" \
    "google-auth==2.49.0"

print("Compatible Gemini SDK installed!")

Compatible Gemini SDK installed!


In [1]:
import google.auth
import google.protobuf
from google import genai
from google.genai import types

print("Gemini SDK imported successfully!")
print("Google Auth:", google.auth.__version__)
print("Protobuf:", google.protobuf.__version__)

Gemini SDK imported successfully!
Google Auth: 2.49.0
Protobuf: 7.36.2


In [4]:
DOCUMENTS = {
    "masala_dosa": (
        "Masala dosa is a popular South Indian breakfast dish. "
        "It is made using a fermented batter of rice and urad dal. "
        "The dosa is cooked until crisp and filled with a spiced potato mixture. "
        "It is commonly served with coconut chutney and sambar."
    ),

    "idli": (
        "Idli is a soft and steamed South Indian breakfast dish. "
        "It is traditionally prepared using fermented rice and urad dal batter. "
        "Idlis are light, fluffy, and easy to digest. "
        "They are usually served with sambar and coconut chutney."
    ),

    "biryani": (
        "Biryani is a popular rice dish prepared with aromatic spices. "
        "It can be made with vegetables, chicken, or other ingredients. "
        "Long-grain basmati rice is commonly used to prepare biryani. "
        "It is often served with raita or a side dish."
    ),

    "paneer_butter_masala": (
        "Paneer butter masala is a popular North Indian curry. "
        "It is prepared using paneer cubes in a creamy tomato-based gravy. "
        "Butter, cream, and spices give the dish its rich flavour. "
        "It is commonly served with naan, roti, or jeera rice."
    )
}

In [6]:
# @title Define a tool
def search_documents(query: str) -> str:
    """Search the document collection for paragraphs relevant to the query."""
    query_lower = query.lower()
    results = []

    for doc_name, content in DOCUMENTS.items():
        words = set(query_lower.split())
        match_count = sum(1 for word in words if word in content.lower())
        if match_count > 0:
            results.append((match_count, doc_name, content))

    results.sort(reverse=True)

    if not results:
        return "No relevant documents found for that query."

    # Return the top two matches
    output = []
    for _, name, content in results[:2]:
        output.append(f"[Source: {name}]\n{content}")
    return "\n\n".join(output)


# Sanity check for regex match.
print(search_documents("minimum of 4"))

[Source: masala_dosa]
Masala dosa is a popular South Indian breakfast dish. It is made using a fermented batter of rice and urad dal. The dosa is cooked until crisp and filled with a spiced potato mixture. It is commonly served with coconut chutney and sambar.

[Source: idli]
Idli is a soft and steamed South Indian breakfast dish. It is traditionally prepared using fermented rice and urad dal batter. Idlis are light, fluffy, and easy to digest. They are usually served with sambar and coconut chutney.


In [13]:
# @title Initialize Gemini for the ReAct agent

import os
import time
from google import genai
from google.genai import types

client = genai.Client(
    api_key=os.environ["GOOGLE_API_KEY"]
)

model_name = "gemini-3.8-flash"

system_instruction = """
You are a chef training at a Michelin-star restaurant.

When you need specific information about ingredients,
cooking temperatures, preparation methods, or cooking times,
use the search_documents tool.

Only answer based on information found in the documents.
If the documents do not contain the answer, say so clearly.
"""

print("Gemini client ready:", model_name)

Gemini client ready: gemini-3.8-flash


In [8]:
# Define the search tool for Gemini

search_tool = types.Tool(
    function_declarations=[
        types.FunctionDeclaration(
            name="search_documents",
            description=(
                "Search the available cooking documents for information "
                "about ingredients, recipes, cooking methods and food."
            ),
            parameters=types.Schema(
                type=types.Type.OBJECT,
                properties={
                    "query": types.Schema(
                        type=types.Type.STRING,
                        description="The keywords or question to search for."
                    )
                },
                required=["query"]
            )
        )
    ]
)

config = types.GenerateContentConfig(
    system_instruction=system_instruction,
    tools=[search_tool],
    automatic_function_calling=types.AutomaticFunctionCallingConfig(
        disable=True
    )
)

print("Search tool configured successfully!")

Search tool configured successfully!


In [16]:
# @title ReAct Agent Loop with Retry Handling

def run_agent(question, max_steps=3):
    print("User:", question)

    messages = [
        types.Content(
            role="user",
            parts=[types.Part(text=question)]
        )
    ]

    for step in range(max_steps):
        response = None

        for attempt in range(3):
            try:
                response = client.models.generate_content(
                    model=model_name,
                    contents=messages,
                    config=config
                )
                break
            except Exception as e:
                if "503" in str(e) and attempt < 2:
                    print("Gemini is busy. Retrying...")
                    time.sleep(10 * (attempt + 1))
                else:
                    print("Gemini request failed:", e)
                    return

        if not response or not response.candidates:
            print("No response generated.")
            return

        candidate = response.candidates[0]
        messages.append(candidate.content)

        function_calls = [
            part.function_call
            for part in candidate.content.parts
            if part.function_call
        ]

        if not function_calls:
            print("\nAgent:", response.text)
            return

        function_responses = []

        for call in function_calls:
            if call.name == "search_documents":
                args = dict(call.args or {})
                result = search_documents(**args)

                print("\nTool:", call.name)
                print("Search results:", result)

                function_responses.append(
                    types.Part.from_function_response(
                        name=call.name,
                        response={"result": result}
                    )
                )

        if not function_responses:
            print("No valid tool response.")
            return

        messages.append(
            types.Content(
                role="user",
                parts=function_responses
            )
        )

    print("Agent stopped: maximum reasoning steps reached.")

In [17]:
# @title Ask follow up questions

questions = [
    "What ingredients are used to prepare masala dosa?",
    "Why are idlis considered easy to digest?",
    "What is the difference between biryani and paneer butter masala?",
    "How many ghosts eat carrots everyday?"
]

for i, q in enumerate(questions):
    if i > 0:
        time.sleep(15)

    print("\n" + "=" * 60)
    print("Question:", q)
    run_agent(q)
    run_agent("What ingredients are used to prepare masala dosa?")


Question: What ingredients are used to prepare masala dosa?
User: What ingredients are used to prepare masala dosa?
Gemini is busy. Retrying...
Gemini is busy. Retrying...

Tool: search_documents
Search results: [Source: masala_dosa]
Masala dosa is a popular South Indian breakfast dish. It is made using a fermented batter of rice and urad dal. The dosa is cooked until crisp and filled with a spiced potato mixture. It is commonly served with coconut chutney and sambar.

[Source: paneer_butter_masala]
Paneer butter masala is a popular North Indian curry. It is prepared using paneer cubes in a creamy tomato-based gravy. Butter, cream, and spices give the dish its rich flavour. It is commonly served with naan, roti, or jeera rice.

Tool: search_documents
Search results: [Source: masala_dosa]
Masala dosa is a popular South Indian breakfast dish. It is made using a fermented batter of rice and urad dal. The dosa is cooked until crisp and filled with a spiced potato mixture. It is commonly s

In [ ]:
# This cell will check if you have added atleast 2 more questions.
assert(len(questions) >= 4)
# This will check if you added a question.
assert("Add a question here." not in questions)